# CSV to LaTeX table

Edit `csv_path`, run the cells, then copy the printed LaTeX table.

`alpha` is the GPR diagonal regularization/noise level, not a kernel parameter, so the caption below only reports the kernel length-scale and Matern `nu` values.

In [ ]:
from pathlib import Path

import pandas as pd

# Change this to the CSV file you want to convert.
csv_path = Path("parity_check_conservation_law_sample.csv")

In [ ]:
df = pd.read_csv(csv_path)
df

In [ ]:
def fmt_error(mean, std):
    return f"{100 * mean:.2f}\\% ({100 * std:.2f}\\%)"


table_df = df.copy()
method_names = {
    "product_gpr": "product",
}
kernel_names = {
    "Matern x Matern": "M x M",
    "RBF x RBF": "R x R",
}
table_df["Method"] = table_df["method"].replace(method_names).str.replace("_", " ", regex=False)
table_df["Kernel"] = table_df["kernel"].replace(kernel_names)
table_df["Method / kernel"] = table_df["Method"] + " / " + table_df["Kernel"]
table_df["Error"] = [
    fmt_error(mean, std)
    for mean, std in zip(table_df["mean_relative_error"], table_df["std_relative_error"])
]

table_df["Result set"] = table_df["split"]
if "ood_filename" in table_df.columns:
    named_ood = table_df["split"].eq("ood") & table_df["ood_filename"].notna()
    table_df.loc[named_ood, "Result set"] = "OOD " + table_df.loc[named_ood, "ood_filename"].astype(str)

table_keys = ["Method / kernel", "Result set"]
duplicate_cells = table_df.duplicated(table_keys, keep=False)
if duplicate_cells.any():
    print("Duplicate CSV rows map to the same table cell; keeping the first row for each cell.")

simple_table = table_df.pivot_table(
    index="Method / kernel",
    columns="Result set",
    values="Error",
    aggfunc="first",
    sort=False,
).reset_index()
simple_table.columns.name = None

def fmt_values(values):
    return ", ".join(f"{value:.4g}" for value in values)


def unique_values(mask, col):
    if col not in df.columns:
        return []
    return df.loc[mask, col].dropna().unique()


def describe_kernel(length_values, nu_values=None):
    if len(length_values) == 0:
        return None
    description = f"length scale={fmt_values(length_values)}"
    if nu_values is not None and len(nu_values) > 0:
        description += f", nu={fmt_values(nu_values)}"
    return description


def describe_group(title, rbf_length_col, matern_length_col, matern_nu_col, method_name):
    method_mask = df["method"].eq(method_name)
    rbf_mask = method_mask & df["kernel"].str.contains("RBF", na=False)
    matern_mask = method_mask & df["kernel"].str.contains("Matern", na=False)

    parts = []
    rbf = describe_kernel(unique_values(rbf_mask, rbf_length_col))
    matern = describe_kernel(
        unique_values(matern_mask, matern_length_col),
        unique_values(matern_mask, matern_nu_col),
    )

    if rbf:
        parts.append(f"RBF {rbf}")
    if matern:
        parts.append(f"Matern {matern}")
    if not parts:
        return None
    return f"{title}: " + "; ".join(parts)


kernel_param_groups = [
    describe_group("Vanilla kernel parameters", "length_scale", "length_scale", "nu", "vanilla"),
    describe_group("Framework 1 kernel parameters", "length_scale", "length_scale", "nu", "framework1"),
    describe_group("Coefficient kernel parameters", "coef_length_scale", "coef_length_scale", "coef_nu", "product_gpr"),
    describe_group("IC kernel parameters", "state_length_scale", "state_length_scale", "state_nu", "product_gpr"),
]
kernel_param_groups = [group for group in kernel_param_groups if group]

caption = "Mean relative error in percent with standard deviation in parentheses."
if kernel_param_groups:
    caption += " " + ". ".join(kernel_param_groups) + "."

preferred_splits = ["test", "ood"]
ood_file_cols = sorted(col for col in simple_table.columns if str(col).startswith("OOD "))
split_cols = [col for col in preferred_splits if col in simple_table.columns]
split_cols += [col for col in ood_file_cols if col not in split_cols]
split_cols += [col for col in simple_table.columns if col not in ["Method / kernel", *split_cols]]
simple_table = simple_table[["Method / kernel", *split_cols]]

def latex_escape(text):
    return str(text).replace("_", "\\_")


def header_label(col):
    if col == "test":
        return "Test"
    if col == "ood":
        return "OOD"
    return str(col).removeprefix("OOD ").removesuffix(".h5")


headers = ["Method / kernel", *[latex_escape(header_label(col)) for col in split_cols]]
alignment = "l" + "c" * len(split_cols)

row_end = " " + "\\" * 2

latex_lines = [
    "\\begin{table}",
    "\\centering",
    "\\scriptsize",
    f"\\begin{{tabular}}{{{alignment}}}",
    "\\hline",
    " & ".join(headers) + row_end,
    "\\hline",
]

for _, row in simple_table.iterrows():
    method = latex_escape(row["Method / kernel"])
    latex_lines.append(" & ".join([method, *[str(row[col]) for col in split_cols]]) + row_end)

latex_lines += [
    "\\hline",
    "\\end{tabular}",
    f"\\caption{{{caption}}}",
    "\\label{tab:csv-results}",
    "\\end{table}",
]

latex_table = "\n".join(latex_lines)

display(simple_table)
print(latex_table)

## Timing table

By default this uses every `*_all_ood.csv` file next to `csv_path`. Edit `timing_csv_paths` to select a different set of PDE result files.

In [ ]:
timing_csv_paths = sorted(csv_path.parent.glob("*_all_ood.csv"))

# Example manual selection:
# timing_csv_paths = [
#     Path("Framework2/results_Conservation_law_sample_no_pca_all_ood.csv"),
#     Path("Framework2/results_DiffReacAdv_sample_no_pca_all_ood.csv"),
# ]

if not timing_csv_paths:
    timing_csv_paths = [csv_path]

timing_csv_paths

In [ ]:
timing_df = pd.concat(
    [pd.read_csv(path).assign(source_csv=path.name) for path in timing_csv_paths],
    ignore_index=True,
)

required_timing_cols = {
    "pde",
    "method",
    "kernel",
    "split",
    "n_eval",
    "train_time_seconds",
    "predict_time_seconds",
}
missing_timing_cols = required_timing_cols.difference(timing_df.columns)
if missing_timing_cols:
    raise ValueError(f"Timing table needs columns: {sorted(missing_timing_cols)}")

timing_df["Method"] = timing_df["method"].replace(method_names).str.replace("_", " ", regex=False)
timing_df["Kernel"] = timing_df["kernel"].replace(kernel_names)
timing_df["Method / kernel"] = timing_df["Method"] + " / " + timing_df["Kernel"]
timing_df["Prediction ms/sample"] = 1000 * timing_df["predict_time_seconds"] / timing_df["n_eval"]

dedupe_cols = ["pde", "Method / kernel", "split"]
if "ood_filename" in timing_df.columns:
    dedupe_cols.append("ood_filename")
timing_df = timing_df.drop_duplicates(dedupe_cols, keep="first")

timing_rows = []
for (pde, method_kernel), group in timing_df.groupby(["pde", "Method / kernel"], sort=True):
    test_rows = group[group["split"].eq("test")]
    ood_rows = group[group["split"].eq("ood")]
    train_rows = test_rows if not test_rows.empty else group
    timing_rows.append({
        "PDE": str(pde).replace("_", " "),
        "Method / kernel": method_kernel,
        "Train time (s)": train_rows["train_time_seconds"].iloc[0],
        "Test pred. (ms/sample)": test_rows["Prediction ms/sample"].mean(),
        "Mean OOD pred. (ms/sample)": ood_rows["Prediction ms/sample"].mean(),
    })

timing_table = pd.DataFrame(timing_rows)

def fmt_time(value):
    return "--" if pd.isna(value) else f"{value:.3g}"


timing_display = timing_table.copy()
for col in ["Train time (s)", "Test pred. (ms/sample)", "Mean OOD pred. (ms/sample)"]:
    timing_display[col] = timing_display[col].map(fmt_time)

timing_headers = list(timing_display.columns)
timing_alignment = "ll" + "c" * (len(timing_headers) - 2)
row_end = " " + "\\" * 2
timing_latex_lines = [
    "\\begin{table}",
    "\\centering",
    "\\scriptsize",
    f"\\begin{{tabular}}{{{timing_alignment}}}",
    "\\hline",
    " & ".join(latex_escape(header) for header in timing_headers) + row_end,
    "\\hline",
]

for _, row in timing_display.iterrows():
    timing_latex_lines.append(" & ".join(latex_escape(row[col]) for col in timing_headers) + row_end)

timing_latex_lines += [
    "\\hline",
    "\\end{tabular}",
    "\\caption{Training time and prediction time per evaluated sample. Mean OOD prediction time is averaged over the OOD files available for each PDE.}",
    "\\label{tab:timing-results}",
    "\\end{table}",
]

timing_latex_table = "\n".join(timing_latex_lines)

display(timing_display)
print(timing_latex_table)